# 06 - Privacy Attack Evaluation
Notebook ini menyiapkan evaluasi MIA sederhana berbasis confidence-threshold sebagai baseline serangan.

In [1]:
from pathlib import Path
import json
import numpy as np

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    return cwd

PROJECT_ROOT = resolve_project_root()
base_path = PROJECT_ROOT / 'outputs' / 'reports' / 'baseline_metrics.json'
dp_path = PROJECT_ROOT / 'outputs' / 'reports' / 'dp_metrics.json'

baseline = json.loads(base_path.read_text(encoding='utf-8')) if base_path.exists() else {}
dp = json.loads(dp_path.read_text(encoding='utf-8')) if dp_path.exists() else {}
print('Baseline metrics loaded:', bool(baseline))
print('DP metrics loaded:', bool(dp))

Baseline metrics loaded: True
DP metrics loaded: True


In [2]:
import sys
import json
import torch
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from models.baseline import create_baseline_model
from evaluation.attack_mia import evaluate_mia
from opacus.validators import ModuleValidator

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load original sequence bundle
artifact = PROJECT_ROOT / 'data' / 'processed' / 'sequence_bundle.npz'
data = np.load(artifact, allow_pickle=True)
X = data['features'].astype(np.float32)
y_raw = data['labels']
sequence_ids = data['sequence_ids']

labels_unique = sorted(set(y_raw.tolist()))
label_to_idx = {label: i for i, label in enumerate(labels_unique)}
y = np.array([label_to_idx[val] for val in y_raw], dtype=np.int64)

# Session splits for Baseline and DP
session_groups = np.asarray([str(sequence_id).split('_')[1] for sequence_id in sequence_ids])
unique_sessions = np.array(sorted(np.unique(session_groups), key=lambda v: int(v)))
train_sessions = unique_sessions[:-2]
test_sessions = unique_sessions[-1:]

train_mask = np.isin(session_groups, train_sessions)
test_mask = np.isin(session_groups, test_sessions)

X_train_sess, y_train_sess = X[train_mask], y[train_mask]
X_test_sess, y_test_sess = X[test_mask], y[test_mask]

def make_loader(x_arr, y_arr, shuffle=False):
    ds = torch.utils.data.TensorDataset(torch.tensor(x_arr), torch.tensor(y_arr))
    return torch.utils.data.DataLoader(ds, batch_size=64, shuffle=shuffle)

train_loader_sess = make_loader(X_train_sess, y_train_sess)
test_loader_sess = make_loader(X_test_sess, y_test_sess)

# Random splits for FL and FL+DP
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
train_loader_rand = make_loader(X_train_rand, y_train_rand)
test_loader_rand = make_loader(X_test_rand, y_test_rand)

results = {}

# 1. MIA Baseline
try:
    model_path = PROJECT_ROOT / 'outputs' / 'models' / 'baseline_lstm.pt'
    checkpoint = torch.load(model_path, map_location=device)
    model = create_baseline_model(input_dim=X.shape[2], num_classes=len(labels_unique), device=device)
    model.load_state_dict(checkpoint['state_dict'])
    results['baseline'] = evaluate_mia(model, train_loader_sess, test_loader_sess, device=device)
    print('MIA Baseline:', results['baseline'])
except Exception as e:
    print('Baseline MIA failed:', e)

# 2. MIA DP
try:
    model_path = PROJECT_ROOT / 'outputs' / 'models' / 'dp_lstm.pt'
    checkpoint = torch.load(model_path, map_location=device)
    model = create_baseline_model(input_dim=X.shape[2], num_classes=len(labels_unique), device=device)
    model = ModuleValidator.fix(model).to(device)
    state_dict = checkpoint['state_dict']
    if any(k.startswith('_module.') for k in state_dict.keys()):
        state_dict = {k.replace('_module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    results['dp'] = evaluate_mia(model, train_loader_rand, test_loader_rand, device=device)
    print('MIA DP:', results['dp'])
except Exception as e:
    print('DP MIA failed:', e)

# 3. MIA FL
try:
    model_path = PROJECT_ROOT / 'outputs' / 'models' / 'fl_lstm.pt'
    checkpoint = torch.load(model_path, map_location=device)
    model = create_baseline_model(input_dim=X.shape[2], num_classes=len(labels_unique), device=device)
    model.load_state_dict(checkpoint['state_dict'])
    results['fl'] = evaluate_mia(model, train_loader_rand, test_loader_rand, device=device)
    print('MIA FL:', results['fl'])
except Exception as e:
    print('FL MIA failed:', e)

# 4. MIA FL+DP
try:
    model_path = PROJECT_ROOT / 'outputs' / 'models' / 'fl_dp_lstm.pt'
    checkpoint = torch.load(model_path, map_location=device)
    model = create_baseline_model(input_dim=X.shape[2], num_classes=len(labels_unique), device=device)
    model = ModuleValidator.fix(model).to(device)
    state_dict = checkpoint['state_dict']
    if any(k.startswith('_module.') for k in state_dict.keys()):
        state_dict = {k.replace('_module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    results['fl_dp'] = evaluate_mia(model, train_loader_rand, test_loader_rand, device=device)
    print('MIA FL+DP:', results['fl_dp'])
except Exception as e:
    print('FL+DP MIA failed:', e)

out_path = PROJECT_ROOT / 'outputs' / 'reports' / 'attack_metrics.json'
out_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
print('Saved attack metrics to:', out_path)


MIA Baseline: {'attack_accuracy': 0.8663287086446104, 'attack_precision': 0.8663287086446104, 'attack_recall': 1.0, 'attack_auc': 0.5181967447919068}


MIA DP: {'attack_accuracy': 0.7999063889538965, 'attack_precision': 0.7999063889538965, 'attack_recall': 1.0, 'attack_auc': 0.49864015412042884}


MIA FL: {'attack_accuracy': 0.7999063889538965, 'attack_precision': 0.7999063889538965, 'attack_recall': 1.0, 'attack_auc': 0.500277854769555}


MIA FL+DP: {'attack_accuracy': 0.7999063889538965, 'attack_precision': 0.7999063889538965, 'attack_recall': 1.0, 'attack_auc': 0.5009582909878558}
Saved attack metrics to: C:\Users\anang\Downloads\Projek Keamanan Informasi\outputs\reports\attack_metrics.json
